# Demo 4 - Demonstração - Aprendizagem Supervisionada

### MBA em AI Engineering & Multi Agents - Machine Learning Foundations & Classic Models.

### Algoritmo para classificação de transações em possíveis fraudes/não fraude.

### Prof. Dr. Ahirton Lopes (profahirton.lopes@fiap.com.br)

Ref. - https://www.kaggle.com/pwnpen/payment

```markdown

Este notebook serve como uma demonstração prática dos conceitos de Aprendizagem Supervisionada, focando na construção e avaliação de modelos de classificação para detecção de fraudes em transações de pagamento. Através deste exemplo, exploraremos as etapas desde o pré-processamento dos dados até a aplicação de algoritmos como Regressão Logística e Árvore de Decisão, e a interpretação de métricas de desempenho.

**Objetivo:** Classificar transações como fraudulentas (1) ou não fraudulentas (0) utilizando diferentes modelos de Machine Learning.

**Dados:** Utilizaremos um dataset de transações de pagamento contendo informações como idade da conta, número de itens, tempo local, método de pagamento e idade do método de pagamento. A coluna 'label' indica se a transação é fraude (1) ou não (0).

```

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score,precision_score,recall_score, confusion_matrix)

In [ ]:
# ALTERNATIVA: se preferir subir o arquivo na mão, descomente as linhas abaixo,
# execute esta célula e selecione o payment_fraud.csv do seu computador.

# from google.colab import files
#
# uploaded = files.upload()
#
# for fn in uploaded.keys():
#     print('Arquivo "{name}" com {length} bytes'.format(
#         name=fn, length=len(uploaded[fn])))


In [3]:
# Baixando a base de dados direto do repositório da disciplina para o Google Colab
# Fonte original: https://www.kaggle.com/pwnpen/payment

!curl -O https://raw.githubusercontent.com/ahirtonlopes/Machine-Learning-Foundations-and-Classic-Models/main/Datasets/payment_fraud.csv


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1444k  100 1444k    0     0  3122k      0 --:--:-- --:--:-- --:--:-- 3127k


In [4]:
# Lendo dados a partir de nosso arquivo .csv
df = pd.read_csv('payment_fraud.csv')

In [5]:
# Amostragem de cabeçalho e cinco primeiras linhas de nosso dataset
df.sample(5)

,accountAgeDays,numItems,localTime,paymentMethod,paymentMethodAgeDays,label
31372,1852,1,4.965339,creditcard,0.014583,0
35635,1370,1,4.836982,storecredit,0.000000,0
28665,2000,1,4.965339,creditcard,270.193750,0
37628,524,1,5.017904,creditcard,0.000000,0
38236,84,1,4.742303,creditcard,0.000000,0


In [6]:
# Verificando linhas e partir do índice de nosso dataset
len(df.index)

39221

In [7]:
# Verificando colunas de nosso dataset
df.columns

Index(['accountAgeDays', 'numItems', 'localTime', 'paymentMethod',
       'paymentMethodAgeDays', 'label'],
      dtype='object')

In [8]:
# Verificando itens únicos de nossa coluna 'paymentMethod' de nosso dataset
(df['paymentMethod'].unique())

array(['paypal', 'storecredit', 'creditcard'], dtype=object)

In [9]:
# Agrupando número de itens de nossa coluna 'numItems' de nosso dataset
df.groupby('numItems').size().reset_index()

,numItems,0
0,1,37398
1,2,1348
2,3,164
3,4,42
4,5,168
5,6,15
6,7,5
7,8,5
8,9,1
9,10,71


In [10]:
# Tratando nossos dados da coluna 'pamentMethod' para formato numérico - Ref. https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html
df_one_hot = pd.get_dummies(df, columns=['paymentMethod'])

In [11]:
df_one_hot.sample(3)

,accountAgeDays,numItems,localTime,paymentMethodAgeDays,label,paymentMethod_creditcard,paymentMethod_paypal,paymentMethod_storecredit
10958,2000,1,4.921318,0.008333,0,True,False,False
34464,927,1,5.017904,0.000000,0,True,False,False
1347,2000,2,4.895263,0.000000,0,True,False,False


In [12]:
# Divisão em sets de treinamento/teste (Regra de Pareto - 80/20 - Ref. https://pt.wikipedia.org/wiki/Princ%C3%ADpio_de_Pareto)
X_train, X_test, y_train, y_test = train_test_split(df_one_hot.drop('label', axis=1), df_one_hot['label'], test_size=0.2, random_state=42)

In [13]:
len(X_train.columns)

7

## Avaliação do Modelo: Compreendendo as Métricas de Desempenho

Após treinar nossos modelos de Machine Learning (Regressão Logística e Árvore de Decisão) para classificar transações como fraude ou não fraude, é crucial avaliar seu desempenho. As métricas de avaliação nos ajudam a entender o quão bem o modelo está realizando suas previsões e quais tipos de erros ele está cometendo.

Para problemas de classificação como o nosso (detecção de fraude), métricas como **Acurácia**, **Precisão** e **Recall** são fundamentais. Além disso, a **Matriz de Confusão** oferece uma visão detalhada dos acertos e erros do modelo.

### Entendendo as Métricas Chave:

*   **Acurácia (Accuracy)**: É a proporção de previsões corretas em relação ao total de previsões. Indica a eficácia geral do modelo.
    *   `Acurácia = (Verdadeiros Positivos + Verdadeiros Negativos) / (Total de Amostras)`

*   **Precisão (Precision)**: Mede a proporção de verdadeiros positivos entre todas as previsões positivas (ou seja, quando o modelo previu 'fraude', quantas vezes estava realmente correto).
    *   É importante quando o custo de um falso positivo é alto (ex: classificar uma transação legítima como fraude, causando inconveniência ao cliente).
    *   `Precisão = Verdadeiros Positivos / (Verdadeiros Positivos + Falsos Positivos)`

*   **Recall (Sensibilidade ou Cobertura)**: Mede a proporção de verdadeiros positivos entre todas as amostras que realmente eram positivas (ou seja, de todas as fraudes reais, quantas o modelo conseguiu detectar).
    *   É importante quando o custo de um falso negativo é alto (ex: não detectar uma fraude real, resultando em perda financeira).
    *   `Recall = Verdadeiros Positivos / (Verdadeiros Positivos + Falsos Negativos)`

*   **Matriz de Confusão (Confusion Matrix)**: Uma tabela que descreve o desempenho de um modelo de classificação em um conjunto de dados de teste para o qual os valores verdadeiros são conhecidos. Ela mostra o número de:
    *   **Verdadeiros Positivos (VP)**: Fraudes corretamente identificadas.
    *   **Verdadeiros Negativos (VN)**: Não-fraudes corretamente identificadas.
    *   **Falsos Positivos (FP)**: Não-fraudes incorretamente identificadas como fraude (Erro Tipo I).
    *   **Falsos Negativos (FN)**: Fraudes incorretamente identificadas como não-fraude (Erro Tipo II).

In [14]:
# Construção de nosso modelo usando Regressão Logística (Ref. https://edisciplinas.usp.br/pluginfile.php/3769787/mod_resource/content/1/09_RegressaoLogistica.pdf)
clf = LogisticRegression(max_iter=20).fit(X_train, y_train)

# Predição em dados de teste
y_pred = clf.predict(X_test)

In [15]:
# Avaliação de Acurácia, Precisão e Recall de nosso modelo
accuracy = accuracy_score(y_pred, y_test)
precision = precision_score(y_pred, y_test)
recall = recall_score(y_pred, y_test)

print("acuracia", accuracy)
print("precisao", precision)
print("recall", recall)

acuracia 1.0
precisao 1.0
recall 1.0


In [16]:
# Utilizando Árvore de Decisão
clf = DecisionTreeClassifier()
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

# Avaliação de Acurácia, Precisão e Recall de nosso modelo
accuracy = accuracy_score(y_pred, y_test)
precision = precision_score(y_pred, y_test)
recall = recall_score(y_pred, y_test)

print("acuracia", accuracy)
print("precisao", precision)
print("recall", recall)

acuracia 1.0
precisao 1.0
recall 1.0


```markdown
### Testando o Modelo com Novos Exemplos

Agora que o modelo de Árvore de Decisão foi treinado, podemos usá-lo para classificar novas transações. Vamos criar alguns exemplos hipotéticos e ver como o modelo os classifica como fraude (1) ou não fraude (0).

Para isso, precisamos garantir que os dados de teste tenham o mesmo formato (as mesmas colunas) que os dados usados para treinar o modelo.
```

In [17]:
# Criando exemplos para teste
# As colunas devem corresponder às do X_train/X_test:
# 'accountAgeDays', 'numItems', 'localTime', 'paymentMethodAgeDays',
# 'paymentMethod_creditcard', 'paymentMethod_paypal', 'paymentMethod_storecredit'

novos_exemplos = pd.DataFrame([
    {
        'accountAgeDays': 10,
        'numItems': 1,
        'localTime': 4.5,
        'paymentMethodAgeDays': 5.0,
        'paymentMethod_creditcard': True,
        'paymentMethod_paypal': False,
        'paymentMethod_storecredit': False
    }, # Exemplo 1: Transação de crédito recente, conta nova, pode ser suspeita (mas vamos ver)
    {
        'accountAgeDays': 1500,
        'numItems': 2,
        'localTime': 4.9,
        'paymentMethodAgeDays': 1000.0,
        'paymentMethod_creditcard': False,
        'paymentMethod_paypal': True,
        'paymentMethod_storecredit': False
    }, # Exemplo 2: Usuário antigo, método de pagamento antigo, parece legítimo
    {
        'accountAgeDays': 5,
        'numItems': 1,
        'localTime': 4.0,
        'paymentMethodAgeDays': 0.1,
        'paymentMethod_creditcard': False,
        'paymentMethod_paypal': False,
        'paymentMethod_storecredit': True
    }, # Exemplo 3: Conta e método de pagamento muito novos, item único, pode ser fraude
    {
        'accountAgeDays': 300,
        'numItems': 1,
        'localTime': 4.7,
        'paymentMethodAgeDays': 250.0,
        'paymentMethod_creditcard': True,
        'paymentMethod_paypal': False,
        'paymentMethod_storecredit': False
    }
])

print("Novos exemplos para classificação:")
display(novos_exemplos)

# Fazendo a predição com o modelo de Árvore de Decisão
predicoes_novos_exemplos = clf.predict(novos_exemplos)

print("\nPrevisões do modelo (0 = Não Fraude, 1 = Fraude):")
for i, predicao in enumerate(predicoes_novos_exemplos):
    print(f"Exemplo {i+1}: {predicao}")


Novos exemplos para classificação:


,accountAgeDays,numItems,localTime,paymentMethodAgeDays,paymentMethod_creditcard,paymentMethod_paypal,paymentMethod_storecredit
0,10,1,4.5,5.0,True,False,False
1,1500,2,4.9,1000.0,False,True,False
2,5,1,4.0,0.1,False,False,True
3,300,1,4.7,250.0,True,False,False



Previsões do modelo (0 = Não Fraude, 1 = Fraude):
Exemplo 1: 0
Exemplo 2: 0
Exemplo 3: 0
Exemplo 4: 0
